In [9]:
import geopandas as gpd
import pandas as pd
import numpy as np

from pathlib import Path

DATA_PATH = Path("./data")

bike_lanes = gpd.read_file(
    DATA_PATH / "popup_bike_lanes.geojson"
)

print("Shape:", bike_lanes.shape)
print("\nColumns:")
print(bike_lanes.columns.tolist())

print("\nCRS:")
print(bike_lanes.crs)

print("\nGeometry types:")
print(bike_lanes.geometry.geom_type.value_counts())

Shape: (354, 9)

Columns:
['fid', 'roadnm', 'side', 'trialdate', 'postdate', 'lga', 'trialinfra', 'postinfra', 'geometry']

CRS:
EPSG:4326

Geometry types:
MultiLineString    354
Name: count, dtype: int64


In [10]:
bike_lanes.describe(include="all")
print(bike_lanes.head(3))
print(
    bike_lanes.geometry.geom_type.value_counts()
)

   fid          roadnm   side trialdate postdate          lga    trialinfra  \
0    2  Charles Street  North    Jul-22      NaN  Maribyrnong  Painted lane   
1    3   Albert Street   West    Jul-22      NaN  Maribyrnong  Painted lane   
2    4   Albert Street   East    Jul-22      NaN  Maribyrnong  Painted lane   

  postinfra                                           geometry  
0       NaN  MULTILINESTRING ((144.89243 -37.80672, 144.892...  
1       NaN  MULTILINESTRING ((144.89601 -37.80827, 144.896...  
2       NaN  MULTILINESTRING ((144.89706 -37.80609, 144.897...  
MultiLineString    354
Name: count, dtype: int64


In [11]:
# Check missing values
missing = (
    bike_lanes.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing)

#Check duplicated
print(
    "Duplicate FIDs:",
    bike_lanes["fid"].duplicated().sum()
)


postdate      229
postinfra     229
trialdate       1
fid             0
side            0
roadnm          0
lga             0
trialinfra      0
geometry        0
dtype: int64
Duplicate FIDs: 1


In [12]:
# rename column
bike_lanes = bike_lanes.rename(
    columns={
        "fid": "intervention_id",
        "roadnm": "road_name",
        "side": "road_side",
        "trialdate": "trial_start",
        "postdate": "trial_end",
        "lga": "lga",
        "trialinfra": "trial_infrastructure",
        "postinfra": "post_infrastructure",
        "route_name": "route_name"
    }
)
print(bike_lanes.columns.tolist())
print(bike_lanes.head(3))

['intervention_id', 'road_name', 'road_side', 'trial_start', 'trial_end', 'lga', 'trial_infrastructure', 'post_infrastructure', 'geometry']
   intervention_id       road_name road_side trial_start trial_end  \
0                2  Charles Street     North      Jul-22       NaN   
1                3   Albert Street      West      Jul-22       NaN   
2                4   Albert Street      East      Jul-22       NaN   

           lga trial_infrastructure post_infrastructure  \
0  Maribyrnong         Painted lane                 NaN   
1  Maribyrnong         Painted lane                 NaN   
2  Maribyrnong         Painted lane                 NaN   

                                            geometry  
0  MULTILINESTRING ((144.89243 -37.80672, 144.892...  
1  MULTILINESTRING ((144.89601 -37.80827, 144.896...  
2  MULTILINESTRING ((144.89706 -37.80609, 144.897...  


In [13]:
# Convert datetime values using explicit month-year format directive (%b-%y)
bike_lanes["trial_start"] = pd.to_datetime(
    bike_lanes["trial_start"],
    format="%b-%y",
    errors="coerce"
)

bike_lanes["trial_end"] = pd.to_datetime(
    bike_lanes["trial_end"],
    format="%b-%y",
    errors="coerce"
)

# Calculate trial duration in days based on normalized month-start dates
bike_lanes["trial_duration_days"] = (
    bike_lanes["trial_end"] - bike_lanes["trial_start"]
).dt.days
print(
    bike_lanes[
        [
            "trial_start",
            "trial_end"
        ]
    ].head()
)

  trial_start  trial_end
0  2022-07-01        NaT
1  2022-07-01        NaT
2  2022-07-01        NaT
3  2022-07-01 2023-07-01
4  2022-07-01 2023-07-01


In [14]:
# Create new colums to claculate the duration of popup bike lands
bike_lanes["trial_duration_days"] = (
    bike_lanes["trial_end"]
    - bike_lanes["trial_start"]
).dt.days



In [15]:
# 1. Ensure target directory exists
PROCESSED_DATA_PATH = Path("./processed_data")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# 2. Define export path for the bike lanes dataset
file_path = PROCESSED_DATA_PATH / "bike_lanes.geojson"

# 3. Export cleaned GeoDataFrame to GeoJSON
bike_lanes.to_file(file_path, driver="GeoJSON")

# 4. Print inspection metrics matching standard signature
print("=" * 60)
print(f" GEODATASET: bike_lanes (Saved to: {file_path})")
print("=" * 60)
print(bike_lanes.columns.to_list())
print("\n--- Head (3) ---")
print(bike_lanes.head(3))
print("\n--- Info ---")
bike_lanes.info()

 GEODATASET: bike_lanes (Saved to: processed_data\bike_lanes.geojson)
['intervention_id', 'road_name', 'road_side', 'trial_start', 'trial_end', 'lga', 'trial_infrastructure', 'post_infrastructure', 'geometry', 'trial_duration_days']

--- Head (3) ---
   intervention_id       road_name road_side trial_start trial_end  \
0                2  Charles Street     North  2022-07-01       NaT   
1                3   Albert Street      West  2022-07-01       NaT   
2                4   Albert Street      East  2022-07-01       NaT   

           lga trial_infrastructure post_infrastructure  \
0  Maribyrnong         Painted lane                 NaN   
1  Maribyrnong         Painted lane                 NaN   
2  Maribyrnong         Painted lane                 NaN   

                                            geometry  trial_duration_days  
0  MULTILINESTRING ((144.89243 -37.80672, 144.892...                  NaN  
1  MULTILINESTRING ((144.89601 -37.80827, 144.896...                  NaN  
2  